In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
import cv2
import zipfile
import hashlib

from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import AdamW, Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.applications import EfficientNetB3
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.losses import CategoricalFocalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.layers import BatchNormalization, GlobalAveragePooling2D, LeakyReLU, AveragePooling2D
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, confusion_matrix

In [3]:
from pathlib import Path
import os
import shutil
import random

Split data into folders

In [4]:
#script_path = os.path.dirname(os.path.abspath(__file__))
script_path = ""
original_folder = os.path.join(script_path, "EuroSAT_RGB")
base_folder = os.path.join(script_path, "EuroSAT_RGB_split")

In [5]:
def split_data():
    train_ratio = 0.7
    val_ratio = 0.15
    random_state = 42
    random.seed(random_state)

    for split in ["train", "validate", "test"]:
        os.makedirs(os.path.join(base_folder, split), exist_ok=True)
    
    for class_name in os.listdir(original_folder):
        class_path = os.path.join(original_folder, class_name)
        if not os.path.isdir(class_path):
            continue

        images = [f for f in os.listdir(class_path) if f.lower().endswith((".jpg", ".png", ".tif"))]
        random.shuffle(images)

        n_total = len(images)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)

        splits = {
            "train": images[:n_train],
            "validate": images[n_train:n_train + n_val],
            "test": images[n_train + n_val:]
        }

        for split_name, split_images in splits.items():
            split_dir = os.path.join(base_folder, split_name, class_name)
            os.makedirs(split_dir, exist_ok=True)

            for img_name in split_images:
                src = os.path.join(class_path, img_name)
                dst = os.path.join(split_dir, img_name)
                shutil.copy(src, dst)

        print(f"{class_name}: {n_total} images split into train/val/test")

In [6]:
#split_data()
# took 3 mins

AnnualCrop: 3000 images split into train/val/test

Forest: 3000 images split into train/val/test

HerbaceousVegetation: 3000 images split into train/val/test

Highway: 2500 images split into train/val/test

Industrial: 2500 images split into train/val/test

Pasture: 2000 images split into train/val/test

PermanentCrop: 2500 images split into train/val/test

Residential: 3000 images split into train/val/test

River: 2500 images split into train/val/test

SeaLake: 3000 images split into train/val/test


In [7]:
train_dir = os.path.join(base_folder, "train")
val_dir = os.path.join(base_folder, "validate")
test_dir = os.path.join(base_folder, "test")

In [8]:
train_datagen = ImageDataGenerator(rescale=1./255)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    shuffle=False 
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    shuffle=False 
)

Found 18900 images belonging to 10 classes.
Found 4050 images belonging to 10 classes.
Found 4050 images belonging to 10 classes.


In [ ]:
# import tensorflow as tf
# print("TensorFlow version:", tf.__version__)
# print("GPUs available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.20.0
GPUs available: []


In [ ]:
# import os, tensorflow as tf
# print(tf.sysconfig.get_build_info()["cuda_version"])
# print(tf.sysconfig.get_build_info()["cudnn_version"])

KeyError: 'cuda_version'

In [ ]:
# from tensorflow.python.platform import build_info
# print(build_info.build_info)


OrderedDict({'is_cuda_build': False, 'is_rocm_build': False, 'is_tensorrt_build': False, 'msvcp_dll_names': 'msvcp140.dll,msvcp140_1.dll'})


In [13]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 16350525004020771610
xla_global_id: -1
]


In [ ]:
class_labels = train_generator.classes
class_weight_dict = dict(
    enumerate(
        compute_class_weight('balanced', classes=np.unique(class_labels), y=class_labels)
    )
)

model = Sequential([
    *[
        layer
        for filters in [64, 128, 256, 512]
        for layer in (
            [Conv2D(filters, 3, activation='relu', padding='same', kernel_regularizer=l2(0.001)),
             BatchNormalization(),
             MaxPooling2D(2)]
        )
    ],
    GlobalAveragePooling2D(),
    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.4),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer=Adam(1e-4),
    loss=CategoricalFocalCrossentropy(gamma=2.0),
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6)
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    class_weight=class_weight_dict,
    callbacks=callbacks
)


c:\Users\S1NdBAD\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50


In [ ]:
def plot_training_curves(model):
    metrics = ['accuracy', 'loss']
    titles = ['Accuracy Curve', 'Loss Curve']
    colors = ['#44916F', '#DAA06D']

    plt.figure(figsize=(10, 4))
    for i, metric in enumerate(metrics, 1):
        plt.subplot(1, 2, i)
        plt.plot(model.history[metric], label=f'Train {metric.title()}', color=colors[0], linewidth=1.5)
        plt.plot(model.history[f'val_{metric}'], label=f'Validation {metric.title()}', color=colors[1], linewidth=1.5)
        plt.xlabel('Epochs')
        plt.ylabel(metric.title())
        plt.title(titles[i-1])
        plt.legend()
        plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_training_curves(model)

In [ ]:
def evaluate_model(model, test_generator):
    y_prob = model.predict(test_generator)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = test_generator.classes
    labels = list(test_generator.class_indices.keys())

    test_loss, test_acc = model.evaluate(test_generator, verbose=0)
    print(f"\nTest Results:\nAccuracy: {test_acc:.4f} | Loss: {test_loss:.4f}")

    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1-score: {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=labels, digits=4))

    for norm, title in [(None, "Confusion Matrix"), ('true', "Normalized Confusion Matrix")]:
        cm = confusion_matrix(y_true, y_pred, normalize=norm)
        fmt = '.2f' if norm else 'd'
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=labels, yticklabels=labels)
        plt.title(title)
        plt.xlabel("Predicted Label")
        plt.ylabel("True Label")
        plt.tight_layout()
        plt.show()

In [ ]:
evaluate_model(model, test_generator)